# NetworkX Solver Comparison

This notebook compares different NetworkX pathfinding solvers on elevation data.
For a random tensor file at a given resolution (default 4096), we:
1. Load the elevation data
2. Generate random start/end markers
3. Run each solver with configurable cost models
4. Compare execution times and paths

New options supported by `generate.py`:
- `cost_model`: elevation, distance, slope_uphill, energy, absolute_elevation
- `graph_backend`: edges (default) or scipy

In [ ]:
import sys
import time
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Add src to path for imports
sys.path.insert(0, str(Path("../src").resolve()))

from passage.generate import find_path

## Configuration

Set the resolution and optionally specify a tensor index. Configure cost model and graph backend for the solver comparison.

In [ ]:
# Configuration
RESOLUTION = 4096  # Resolution to test (64, 128, 256, 512, 1024, 2048, 4096)
TENSOR_INDEX = None  # Set to integer to use specific tensor, or None for random

# Cost model and graph backend
COST_MODEL = "elevation"  # elevation, distance, slope_uphill, energy, absolute_elevation
GRAPH_BACKEND = "edges"  # edges (fast edge list) or scipy (sparse + from_scipy_sparse_array)

# Solver configurations to compare
# Edge weights account for physical distance: 30m cardinal, 42.43m diagonal
SOLVERS = [
    {"name": "astar", "heuristic": "euclidean", "label": "A* (Euclidean)"},
    {"name": "astar", "heuristic": "manhattan", "label": "A* (Manhattan)"},
    {"name": "dijkstra", "label": "Dijkstra"},
    {"name": "dijkstra_bi", "label": "Dijkstra (Bidirectional)"},
    {"name": "bellman_ford", "label": "Bellman-Ford"},
    {"name": "unweighted", "label": "Unweighted (BFS)"},
]

# Common solver parameters
WEIGHT = 30.0
DIAGONAL_MOVEMENT = "always"

## Load Tensor Data

Find available tensors at the specified resolution and load one.

In [ ]:
# Find tensor files
tensors_dir = Path(f"../outputs/dataset/{RESOLUTION}/tensors")
metadata_dir = Path(f"../outputs/dataset/{RESOLUTION}/metadata")

if not tensors_dir.exists():
    raise FileNotFoundError(
        f"No tensors found at {tensors_dir}. Generate samples first with: uv run passage generate --resolution {RESOLUTION}"
    )

tensor_files = sorted(tensors_dir.glob("*.npy"))
print(f"Found {len(tensor_files)} tensor files at resolution {RESOLUTION}")

# Select tensor
if TENSOR_INDEX is not None:
    tensor_path = tensors_dir / f"{TENSOR_INDEX:010d}.npy"
else:
    tensor_path = random.choice(tensor_files)

print(f"Using tensor: {tensor_path.name}")

# Load tensor (shape: H, W, 2) - channels: elevation, markers
tensor = np.load(tensor_path)
print(f"Tensor shape: {tensor.shape}")

# Extract elevation channel (normalized 0-1) and markers
elevation = tensor[:, :, 0]
markers = tensor[:, :, 1]

# Load metadata for reference
metadata_path = metadata_dir / tensor_path.with_suffix(".yaml").name
with open(metadata_path) as f:
    metadata = yaml.safe_load(f)

crop_min = float(metadata["crop_min_elevation"])
crop_max = float(metadata["crop_max_elevation"])
elevation_raw = elevation * (crop_max - crop_min) + crop_min

print("\nMetadata:")
print(f"  Target elevation: {metadata['target_elevation']:.1f}m")
print(f"  Crop elevation range: {crop_min:.1f}m - {crop_max:.1f}m")
print(f"  Solver: {metadata.get('solver', {}).get('name', 'N/A')}")

## Extract Markers

Get start and end positions from the tensor's marker channel.

In [ ]:
# Find marker positions
marker_positions = np.argwhere(markers > 0.5)
print(f"Found {len(marker_positions)} markers")

if len(marker_positions) >= 2:
    start = tuple(marker_positions[0])
    end = tuple(marker_positions[1])
else:
    # Generate random markers if not found
    h, w = elevation.shape
    start = (random.randint(0, h - 1), random.randint(0, w - 1))
    end = (random.randint(0, h - 1), random.randint(0, w - 1))

print(f"Start: {start}")
print(f"End: {end}")
print(f"Euclidean distance: {np.sqrt((start[0] - end[0]) ** 2 + (start[1] - end[1]) ** 2):.1f} pixels")

## Run Solver Comparison

Test each solver and record timing and path length.

In [ ]:
# Use raw elevation in meters for cost calculations
elevation_input = elevation_raw

results = []

for solver_info in SOLVERS:
    solver_config = {
        "name": solver_info["name"],
        "diagonal_movement": DIAGONAL_MOVEMENT,
        "weight": WEIGHT,
        "cost_model": COST_MODEL,
        "graph_backend": GRAPH_BACKEND,
    }
    if "heuristic" in solver_info:
        solver_config["heuristic"] = solver_info["heuristic"]

    print(f"\nRunning {solver_info['label']}...")
    print(f"  cost_model={COST_MODEL}, graph_backend={GRAPH_BACKEND}")

    # Time the full find_path call (includes graph construction)
    t0 = time.time()
    path = find_path(elevation_input, start, end, solver_config)
    total_time = time.time() - t0

    results.append(
        {
            "solver": solver_info["label"],
            "path_length": len(path),
            "time_ms": total_time * 1000,
            "cost_model": COST_MODEL,
            "graph_backend": GRAPH_BACKEND,
            "path": path,
        }
    )

    print(f"  Path length: {len(path)} pixels")
    print(f"  Time: {total_time * 1000:.1f} ms")

print("\n" + "=" * 50)
print("Comparison complete!")

## Results Summary

Compare all solvers in a table.

In [ ]:
# Create results DataFrame
df = pd.DataFrame(
    [{"Solver": r["solver"], "Path Length": r["path_length"], "Time (ms)": f"{r['time_ms']:.1f}"} for r in results]
)

# Sort by time
df = df.sort_values("Time (ms)")
df = df.reset_index(drop=True)

print(f"Results for {RESOLUTION}x{RESOLUTION} resolution:")
print(f"Start: {start}, End: {end}")
print()
df

## Timing Bar Chart

In [ ]:
# Bar chart of execution times
fig, ax = plt.subplots(figsize=(10, 5))

solvers = [r["solver"] for r in results]
times = [r["time_ms"] for r in results]

bars = ax.barh(solvers, times, color="steelblue")
ax.set_xlabel("Time (ms)")
ax.set_title(f"Solver Comparison - {RESOLUTION}x{RESOLUTION} Resolution")
ax.invert_yaxis()

# Add time labels
for bar, t in zip(bars, times):
    ax.text(
        bar.get_width() + max(times) * 0.01, bar.get_y() + bar.get_height() / 2, f"{t:.1f} ms", va="center", fontsize=9
    )

plt.tight_layout()
plt.show()

## Path Visualization

Visualize paths from different solvers overlaid on the elevation map.

In [ ]:
# Visualize paths
n_solvers = len(results)
cols = min(3, n_solvers)
rows = (n_solvers + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if n_solvers == 1:
    axes = np.array([axes])
axes = axes.flatten()

# Color map for elevation
for idx, (ax, result) in enumerate(zip(axes, results)):
    # Show elevation
    ax.imshow(elevation, cmap="terrain", alpha=0.7)

    # Draw path
    path = result["path"]
    if len(path) > 0:
        path_arr = np.array(path)
        ax.plot(path_arr[:, 1], path_arr[:, 0], "r-", linewidth=1, alpha=0.8, label="Path")

    # Mark start and end
    ax.scatter([start[1]], [start[0]], c="green", s=50, marker="o", label="Start", zorder=5)
    ax.scatter([end[1]], [end[0]], c="blue", s=50, marker="x", label="End", zorder=5)

    ax.set_title(f"{result['solver']}\n{result['time_ms']:.1f}ms, {result['path_length']} px")
    ax.axis("off")

# Hide unused axes
for ax in axes[n_solvers:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## Path Comparison Detail

Compare path differences between solvers.

In [ ]:
# Calculate path statistics
print("Path Statistics:")
print("=" * 60)

for result in results:
    path = result["path"]
    if len(path) > 1:
        # Calculate total path cost (sum of elevation along path)
        elevations = [elevation_raw[p[0], p[1]] for p in path]
        avg_elevation = np.mean(elevations)
        elevation_variance = np.var(elevations)

        # Calculate path straightness (euclidean / path length)
        euclidean = np.sqrt((start[0] - end[0]) ** 2 + (start[1] - end[1]) ** 2)
        straightness = euclidean / len(path) if len(path) > 0 else 0

        print(
            f"{result['solver']:25s} | Length: {len(path):5d} | "
            f"Avg Elev: {avg_elevation:7.1f}m | Straightness: {straightness:.3f}"
        )
    else:
        print(f"{result['solver']:25s} | No path found")

## Recommendations

Based on the comparison:

- **A* with Euclidean heuristic**: Best overall balance of speed and optimality for 8-connectivity grids
- **A* with Manhattan heuristic**: Better for 4-connectivity (no diagonal moves)
- **Dijkstra**: Guaranteed optimal but slower than A* (no heuristic guidance)
- **Bellman-Ford**: Only needed for negative edge weights (slowest)
- **Unweighted (BFS)**: Fastest but ignores elevation costs (shortest path by number of steps)

Tip: try different `cost_model` settings to bias the path (e.g., `slope_uphill` to avoid climbing).